# Clase 1.5 — Perceptrón y Funciones de Activación
### Versión corregida (v2)

---
## Proceso del perceptrón

```
  X1 ──(W1)──┐
              ├──> h = X1·W1 + X2·W2 + b ──> f(h) ──> salida
  X2 ──(W2)──┘
              + b (bias)
```

## ¿Cómo elegir W1, W2 y b? (proceso mental para fuerza bruta)

1. Ver la tabla de verdad: ¿cuándo quiero salida 1? ¿cuándo 0?
2. Si subir X1 debe subir la salida → W1 **positivo**. Si debe bajarla → W1 **negativo**.
3. El bias mueve el umbral: si necesitas que (0,0) dé 1 → b debe ser positivo.
4. Calcular b despejando del caso más claro y verificar todas las filas.

## ¿Qué función de activación usar?

| Salida esperada | Función | Por qué |
|----------------|---------|--------|
| Exactamente 0 o 1 | **escalon_unitario** | Corta en 0, da solo 0 o 1 |
| Número continuo cualquiera | **lineal** | f(h) = h, sin transformar |
| Probabilidad 0.0 a 1.0 | **sigmoide** | Convierte a rango (0,1) |
| Solo positivos | **relu** | Corta negativos a 0 |

In [1]:
# ═══════════════════════════════════════════════════════
# CELDA 1 — Imports (path relativo — funciona en Windows y Linux)
# ═══════════════════════════════════════════════════════
import sys, os
import numpy as np

# Ruta relativa al machote (sube dos niveles desde este archivo)
# Si el notebook está en Modulo-3/Clases/ → sube a Modulo-3 → sube a raíz → entra a Machote
RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

# Si lo anterior da error, descomenta UNA de estas líneas:
# sys.path.append(r"D:/Proyectos/Diplomado-RNA/Machote")  # Windows
# sys.path.append("/home/usr-lbr-maq20/Documentos/Diplomado-RNA/Machote")  # Linux

print("Imports listos")

Imports listos


In [ ]:
# ═══════════════════════════════════════════════════════
# CELDA 2 — Funciones de activación
# ═══════════════════════════════════════════════════════

def escalon_unitario(x):
    """
    CUÁNDO USAR: cuando la salida debe ser exactamente 0 o 1.
    Devuelve 1 si x >= 0, sino 0.
    Usada en: compuertas lógicas, clasificación binaria simple.
    """
    # float(escalar) convierte arrays 0-d o 1-elemento a número Python
    # para evitar errores con numpy 2.0+
    val = float(x.flat[0]) if hasattr(x, 'flat') else float(x)
    return 1 if val >= 0 else 0

def sigmoide(x):
    """
    CUÁNDO USAR: cuando necesitas probabilidad entre 0 y 1.
    Devuelve 1 / (1 + e^(-x)), rango (0, 1).
    Usada en: clasificación binaria, capas de salida.
    """
    return 1 / (1 + np.exp(-x))

def tangente_hiperbolica(x):
    """
    CUÁNDO USAR: similar a sigmoide pero centrada en 0.
    Devuelve valor entre -1 y 1.
    Usada en: capas ocultas de redes neuronales.
    """
    return np.tanh(x)

def relu(x):
    """
    CUÁNDO USAR: cuando la salida debe ser positiva o cero.
    Devuelve x si x > 0, sino 0.
    Usada en: capas ocultas de redes profundas (la más común hoy en día).
    NOTA: usa np.maximum para evitar problemas con arrays numpy.
    """
    return float(np.maximum(0, x).flat[0]) if hasattr(x, 'flat') else max(0, float(x))

def lineal(x):
    """
    CUÁNDO USAR: cuando la salida es un número continuo sin límite de rango.
    Devuelve x sin modificarlo — f(h) = h.
    Usada en: regresión (predecir precios, velocidades, temperaturas).
    """
    return float(x.flat[0]) if hasattr(x, 'flat') else float(x)

print("Funciones de activación definidas")
print()
# Mini test para verificar que todas funcionan
print("Test rápido:")
print(f"  escalon_unitario(-0.5) = {escalon_unitario(-0.5)} (esperado 0)")
print(f"  escalon_unitario(+0.5) = {escalon_unitario(+0.5)} (esperado 1)")
print(f"  relu(-3)               = {relu(-3)} (esperado 0)")
print(f"  relu(+3)               = {relu(+3)} (esperado 3)")
print(f"  lineal(350)            = {lineal(350)} (esperado 350)")

Funciones de activación definidas

Test rápido:
  escalon_unitario(-0.5) = 0 (esperado 0)
  escalon_unitario(+0.5) = 1 (esperado 1)
  relu(-3)               = 0 (esperado 0)
  relu(+3)               = 3.0 (esperado 3)
  lineal(350)            = 350.0 (esperado 350)


In [3]:
# ═══════════════════════════════════════════════════════
# CELDA 3 — Helper para mostrar resultados
# (versión robusta — funciona con numpy 2.0+)
# ═══════════════════════════════════════════════════════

def to_scalar(val):
    """Convierte cualquier valor numpy a float Python. 
    """
    if hasattr(val, 'flat'):
        return float(val.flat[0])
    return float(val)

def mostrar_resultados(nombre, inputs, targets, outputs):
    """
    Imprime tabla de resultados con ✅/❌.
    Versión robusta que maneja arrays numpy de cualquier forma.
    """
    print(f"\n{'═'*60}")
    print(f"  Compuerta {nombre}")
    print(f"{'═'*60}")
    print(f"  {'X1':>4} {'X2':>4} {'h':>8} {'f(h)':>8} {'Target':>8}  OK")
    print(f"  {'-'*50}")

    correctos = 0
    for i in range(len(outputs)):
        x1  = to_scalar(inputs[i][0])
        x2  = to_scalar(inputs[i][1])
        h   = to_scalar(outputs[i][0])
        fh  = to_scalar(outputs[i][1])
        tgt = to_scalar(targets[i][0])

        ok = '✅' if abs(fh - tgt) < 1e-9 else '❌'
        if ok == '✅':
            correctos += 1

        print(f"  {int(x1):>4} {int(x2):>4} {h:>8.2f} {fh:>8.2f} {tgt:>8.2f}  {ok}")

    print(f"  {'-'*50}")
    if correctos == len(outputs):
        print(f"  Resultado: {correctos}/{len(outputs)} correctos ✅ Compuerta correcta")
    else:
        print(f"  Resultado: {correctos}/{len(outputs)} correctos ❌ Los pesos no la implementan")

print("Helper listo")

Helper listo


In [4]:
# ═══════════════════════════════════════════════════════
# CELDA 4 — Datos de entrada y tablas de verdad
# ═══════════════════════════════════════════════════════

# Las 4 combinaciones posibles de 2 entradas binarias
inputs = np.array([[0,0],[0,1],[1,0],[1,1]])

# Tablas de verdad — lo que DEBE dar cada compuerta
Salida_OR   = np.array([[0],[1],[1],[1]])  # 1 cuando al menos una entrada es 1
Salida_AND  = np.array([[0],[0],[0],[1]])  # 1 solo cuando AMBAS son 1
Salida_NOR  = np.array([[1],[0],[0],[0]])  # 1 solo cuando AMBAS son 0 (inverso de OR)
Salida_NAND = np.array([[1],[1],[1],[0]])  # 0 solo cuando AMBAS son 1 (inverso de AND)
Salida_NOT  = np.array([[1],[0],[0],[0]])  # invierte: 0→1, 1→0
Salida_XOR  = np.array([[0],[1],[1],[0]])  # 1 cuando las entradas son DIFERENTES

print("Datos listos")

Datos listos


---
## Compuerta OR
**Razonamiento:** Necesitamos salida 1 cuando CUALQUIER entrada es 1.
- W1=1, W2=1 → cualquier entrada empuja h hacia positivo
- b=-0.5 → cuando ambas son 0: h=0+0-0.5=-0.5 → 0 ✓
- Caso crítico (0,1): h=0+1-0.5=+0.5 → 1 ✓

⚠️ El notebook de clase tenía W2=-1 aquí, lo que era incorrecto.

In [5]:
outputs = []
targets = Salida_OR
weights = np.array([[1],[1]])  # ambos positivos: cualquier 1 activa la salida
bias    = -0.5                 # umbral bajo: basta con una entrada en 1

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias  # h = X1*W1 + X2*W2 + b
    y = escalon_unitario(h)                 # salida 0 o 1 (clasificación binaria)
    outputs.append([h, y])

mostrar_resultados("OR", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta OR
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0    -0.50     0.00     0.00  ✅
     0    1     0.50     1.00     1.00  ✅
     1    0     0.50     1.00     1.00  ✅
     1    1     1.50     1.00     1.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta AND
**Razonamiento:** Necesitamos salida 1 solo cuando AMBAS son 1.
- W1=1, W2=1 → la suma máxima de las entradas es 2
- b=-1.5 → umbral alto: h(0,1)=1-1.5=-0.5→0 ✓ y h(1,1)=2-1.5=+0.5→1 ✓

In [6]:
outputs = []
targets = Salida_AND
weights = np.array([[1],[1]])  # mismos que OR
bias    = -1.5                 # umbral MÁS alto: necesitas AMBAS entradas en 1

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)
    outputs.append([h, y])

mostrar_resultados("AND", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta AND
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0    -1.50     0.00     0.00  ✅
     0    1    -0.50     0.00     0.00  ✅
     1    0    -0.50     0.00     0.00  ✅
     1    1     0.50     1.00     1.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta NOR
**Razonamiento:** Lo opuesto de OR. Salida 1 solo cuando AMBAS son 0.
- Cualquier entrada en 1 debe apagar la salida → pesos negativos
- b=+0.5 → cuando ambas son 0: h=0+0+0.5=+0.5→1 ✓
- Caso (0,1): h=0+(-1)+0.5=-0.5→0 ✓

In [7]:
outputs = []
targets = Salida_NOR
weights = np.array([[-1],[-1]])  # negativos: cualquier 1 en entrada APAGA la salida
bias    = 0.5                    # positivo: cuando todo es 0, b asegura salida 1

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)
    outputs.append([h, y])

mostrar_resultados("NOR", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta NOR
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0     0.50     1.00     1.00  ✅
     0    1    -0.50     0.00     0.00  ✅
     1    0    -0.50     0.00     0.00  ✅
     1    1    -1.50     0.00     0.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta NAND
**Razonamiento:** Lo opuesto de AND. Salida 0 solo cuando AMBAS son 1.
- Pesos negativos como NOR, pero bias mayor para que (0,1) y (1,0) sigan dando 1.
- b=+1.5 → h(1,1)=-1-1+1.5=-0.5→0 ✓ y h(0,1)=0-1+1.5=+0.5→1 ✓


In [8]:
outputs = []
targets = Salida_NAND  # ← CORRECCIÓN: era Salida_NOR en la clase
weights = np.array([[-1],[-1]])
bias    = 1.5  # más alto que NOR: permite que (0,1) y (1,0) pasen

for i in range(targets.shape[0]):
    h = np.dot(inputs[i], weights) + bias
    y = escalon_unitario(h)
    outputs.append([h, y])

mostrar_resultados("NAND", inputs, targets, outputs)


════════════════════════════════════════════════════════════
  Compuerta NAND
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    0     1.50     1.00     1.00  ✅
     0    1     0.50     1.00     1.00  ✅
     1    0     0.50     1.00     1.00  ✅
     1    1    -0.50     0.00     0.00  ✅
  --------------------------------------------------
  Resultado: 4/4 correctos ✅ Compuerta correcta


---
## Compuerta NOT con LOGIC 1
**Razonamiento:** X1=A (la entrada real), X2=siempre 1 (LOGIC 1).
- W1=-1 → cuando A sube a 1, h baja (queremos que salga 0 cuando A=1)
- W2=0.5 → LOGIC 1 aporta +0.5 constante
- b=0 → h(A=0)=0+0.5=+0.5→1 ✓ y h(A=1)=-1+0.5=-0.5→0 ✓

**¿Por qué escalón y no relu?**
Porque la salida de NOT debe ser exactamente 0 o 1 — es clasificación binaria.
ReLU daría 0.5 para A=0 (porque h=+0.5 → relu devuelve 0.5, no 1).

In [9]:
# X2 siempre vale 1 (LOGIC 1 fijo)
inputs_not = np.array([[0,1],[0,1],[1,1],[1,1]])
outputs = []
targets = Salida_NOT

weights = np.array([[-1],[0.5]])
bias    = 0

for i in range(targets.shape[0]):
    h = np.dot(inputs_not[i], weights) + bias
    y = escalon_unitario(h)  # ← escalon, NO relu (la salida debe ser 0 o 1 exacto)
    outputs.append([h, y])

mostrar_resultados("NOT (X2=LOGIC 1)", inputs_not, targets, outputs)

print()
print("¿Por qué NO usar relu aquí?")
print("  relu(+0.5) = 0.5  ← NO es 1, la compuerta fallaría")
print("  escalon(+0.5) = 1 ← correcto para NOT")


════════════════════════════════════════════════════════════
  Compuerta NOT (X2=LOGIC 1)
════════════════════════════════════════════════════════════
    X1   X2        h     f(h)   Target  OK
  --------------------------------------------------
     0    1     0.50     1.00     1.00  ✅
     0    1     0.50     1.00     0.00  ❌
     1    1    -0.50     0.00     0.00  ✅
     1    1    -0.50     0.00     0.00  ✅
  --------------------------------------------------
  Resultado: 3/4 correctos ❌ Los pesos no la implementan

¿Por qué NO usar relu aquí?
  relu(+0.5) = 0.5  ← NO es 1, la compuerta fallaría
  escalon(+0.5) = 1 ← correcto para NOT


---
## Compuerta XOR — Por qué siempre falla
XOR no es linealmente separable: ninguna línea recta separa los 1s de los 0s.
Cualquier combinación de W1, W2, b fallará en al menos una fila.

In [10]:
print("Probando XOR con varios pesos — todos fallan:")

intentos = [
    ("W=[1,1]   b=-0.5",  np.array([[1],[1]]),    -0.5),
    ("W=[-1,1]  b=-0.5",  np.array([[-1],[1]]),   -0.5),
    ("W=[1,-1]  b=-0.5",  np.array([[1],[-1]]),   -0.5),
    ("W=[-1,-1] b=0.5",   np.array([[-1],[-1]]),   0.5),
]

for nombre, w, b in intentos:
    outputs = []
    for i in range(Salida_XOR.shape[0]):
        h = np.dot(inputs[i], w) + b
        y = escalon_unitario(h)
        outputs.append([h, y])

    correctos = sum(
        1 for i in range(len(outputs))
        if abs(to_scalar(outputs[i][1]) - to_scalar(Salida_XOR[i][0])) < 1e-9
    )
    print(f"  {nombre:22s} → {correctos}/4 " + ("✅" if correctos == 4 else "❌ falla"))

print()
print("Conclusión: XOR requiere Perceptrón Multicapa (MLP).")

Probando XOR con varios pesos — todos fallan:
  W=[1,1]   b=-0.5       → 3/4 ❌ falla
  W=[-1,1]  b=-0.5       → 3/4 ❌ falla
  W=[1,-1]  b=-0.5       → 3/4 ❌ falla
  W=[-1,-1] b=0.5        → 1/4 ❌ falla

Conclusión: XOR requiere Perceptrón Multicapa (MLP).


---
## Ejercicio: Control de Velocidad

| Altura Elevada | Aceleración Alta | Aumento de Velocidad |
|:--------------:|:----------------:|:--------------------:|
| 0 | 0 | **350** |
| 0 | 1 | **200** |
| 1 | 0 | **200** |
| 1 | 1 | **50**  |

**¿Por qué NO usar escalón?** → El escalón solo da 0 o 1. Las salidas son 350, 200, 50.

**¿Por qué usar función lineal?** → La salida es un número continuo, sin límite de rango. f(h) = h.

**Cómo calcular los pesos:**
```
h(0,0) = b = 350              → b = 350
h(0,1) = W2 + 350 = 200       → W2 = -150
h(1,0) = W1 + 350 = 200       → W1 = -150
h(1,1) = -150 + (-150) + 350 = 50  ← verificación ✅
```

In [11]:
inputs_vel  = np.array([[0,0],[0,1],[1,0],[1,1]])
targets_vel = np.array([[350],[200],[200],[50]])

weights_vel = np.array([[-150],[-150]])  # cada condición reduce velocidad en 150
bias_vel    = 350                        # velocidad base cuando no hay condiciones

outputs_vel = []
for i in range(targets_vel.shape[0]):
    h = np.dot(inputs_vel[i], weights_vel) + bias_vel
    y = lineal(h)  # ← LINEAL porque la salida son números continuos (350, 200, 50)
    outputs_vel.append([h, y])

# Mostrar resultados
print(f"{'═'*65}")
print(f"  Control de Velocidad | W1=-150, W2=-150, b=350 | f=lineal")
print(f"{'═'*65}")
print(f"  {'Altura':>8} {'Acel':>6} {'h':>8} {'Salida':>8} {'Target':>8}  OK")
print(f"  {'-'*55}")

correctos = 0
for i in range(len(outputs_vel)):
    x1  = int(inputs_vel[i][0])
    x2  = int(inputs_vel[i][1])
    h   = to_scalar(outputs_vel[i][0])
    fh  = to_scalar(outputs_vel[i][1])
    tgt = to_scalar(targets_vel[i][0])
    ok  = '✅' if abs(fh - tgt) < 1e-9 else '❌'
    if ok == '✅': correctos += 1
    print(f"  {x1:>8} {x2:>6} {h:>8.0f} {fh:>8.0f} {tgt:>8.0f}  {ok}")

print(f"  {'-'*55}")
print(f"  Resultado: {correctos}/4 correctos ✅")

═════════════════════════════════════════════════════════════════
  Control de Velocidad | W1=-150, W2=-150, b=350 | f=lineal
═════════════════════════════════════════════════════════════════
    Altura   Acel        h   Salida   Target  OK
  -------------------------------------------------------
         0      0      350      350      350  ✅
         0      1      200      200      200  ✅
         1      0      200      200      200  ✅
         1      1       50       50       50  ✅
  -------------------------------------------------------
  Resultado: 4/4 correctos ✅
